In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

# Import Bayesian Search from scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer






In [6]:
# ==========================================
# 1. SELECT PARAMETER (FULLY DYNAMIC)
# Options: 'iddq_uA' | 'leakage_current_nA' | 'propagation_delay_ns'
# ==========================================
PARAM_PREFIX = 'iddq_uA'

col_0h = f'{PARAM_PREFIX}_0h'
col_24h = f'{PARAM_PREFIX}_24h'
col_168h = f'{PARAM_PREFIX}_168h'

df = pd.read_csv('clean_burnin_dataset.csv')

# Drop NaNs specific to active parameter features & target
clean_df = df.dropna(subset=[col_0h, col_24h, col_168h]).copy()

# Feature engineering
clean_df['drift_0_to_24'] = clean_df[col_24h] - clean_df[col_0h]
clean_df['ratio_24_to_0'] = clean_df[col_24h] / (clean_df[col_0h] + 1e-6)

X = clean_df[[col_0h, col_24h, 'drift_0_to_24', 'ratio_24_to_0']]
y = clean_df[col_168h]

# Log transformation on target (no data leakage)
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)


In [7]:
# ==========================================
# 2. BAYESIAN OPTIMIZATION SEARCH SPACE
# ==========================================
search_space = {
    'n_estimators': Integer(50, 300),
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
    'subsample': Real(0.6, 1.0)
}

opt = BayesSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    search_spaces=search_space,
    n_iter=15,  # Number of Bayesian optimization iterations
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

# Fit Bayesian Search
opt.fit(X_train, y_train_log)

# Best estimator found by Bayesian optimization
best_model = opt.best_estimator_

In [8]:
# ==========================================
# 3. EVALUATION & DRIFT PREDICTION
# ==========================================
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log) 
y_test = np.expm1(y_test_log)

r2_log = r2_score(y_test_log, y_pred_log)
 

# Metric splits
overall_mae = mean_absolute_error(y_test, y_pred)
normal_mask = y_test < 100
normal_mae = mean_absolute_error(y_test[normal_mask], y_pred[normal_mask])

print(f"=== Bayesian Optimization Results ({PARAM_PREFIX}) ===")
print(f"Best Hyperparameters: {opt.best_params_}")
print(f"Log Scale R^2       : {r2_log:.4f}")
 
print(f"Normal Parts MAE    : {normal_mae:.4f}")
print(f"Overall MAE         : {overall_mae:.4f}")

=== Bayesian Optimization Results (iddq_uA) ===
Best Hyperparameters: OrderedDict({'learning_rate': 0.114012258603381, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.9211059124625243})
Log Scale R^2       : 0.9906
Normal Parts MAE    : 0.6105
Overall MAE         : 0.6105


In [9]:
# ==========================================
# 4. CONSTRUCT & PRINT PREDICTION TABLE
# ==========================================
# Construct DataFrame attached to original test index
results_df = X_test.copy()
results_df['Actual_168h'] = y_test
results_df['Predicted_168h'] = y_pred
results_df['Absolute_Error'] = (results_df['Actual_168h'] - results_df['Predicted_168h']).abs()

# Define table columns to display
table_cols = [col_0h, col_24h, 'Actual_168h', 'Predicted_168h', 'Absolute_Error']

print(f"=== Predictions Table ({PARAM_PREFIX}) ===")
print(results_df[table_cols].head(20).to_string())

=== Predictions Table (iddq_uA) ===
      iddq_uA_0h  iddq_uA_24h  Actual_168h  Predicted_168h  Absolute_Error
592       8.6544       8.9950       8.9000        9.216225        0.316225
4397      7.2204       7.3659       8.4046        7.638649        0.765951
5794     12.4015      13.2515      13.7503       13.255399        0.494901
3296      6.8493       7.1121       7.1421        7.326740        0.184640
5309     16.1637      15.7050      18.3721       17.222395        1.149705
4139     10.8418      10.6959      11.9791       11.620451        0.358649
6009     16.8660      18.0930      20.3628       18.381835        1.980965
3033      5.3782       5.0101       5.7115        5.838966        0.127466
7361     12.0877      12.2925      15.4607       12.634591        2.826109
7296     10.8656      11.3351      11.0292       11.737685        0.708485
960      12.8472      13.4564      14.6374       13.763553        0.873847
8163     11.4110      11.7947      13.6828       12.275331      